# DiffusionBlocks++ on Colab: train once, compare Euler and midpoint

This notebook trains a **DiffusionBlocks (`dblock`)** classifier on a Colab GPU, copies its checkpoint to Google Drive, then evaluates that same checkpoint with Euler and midpoint. The training loss does not depend on the inference sampler, so you do not need to train a separate model for each method.

Set **Runtime → Change runtime type → GPU** before starting. The default starter run uses 5 `num_epochs`; for `dblock`, the training script multiplies this by `num_blocks` (3 by default), so that is 15 Lightning trainer epochs. Increase it for a more meaningful accuracy comparison.

The evaluation section sweeps Euler and midpoint over configurable inference sizes, saves logs and a CSV to Drive, and plots test accuracy. **Already have a checkpoint?** Run the GPU, Drive, repository, dependency, and W&B setup cells, skip training/checkpoint copying, and set `CHECKPOINT` and `NUM_BLOCKS` in the sweep configuration.

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select a GPU runtime, then rerun this cell.")
print("GPU:", torch.cuda.get_device_name(0))

## Connect Drive and get the code

The checkpoint will be copied to Drive so it survives the temporary Colab runtime.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%cd /content
!test -d /content/DiffusionBlocksPlusPlus || git clone --branch heun-method https://github.com/nhford/DiffusionBlocksPlusPlus.git
%cd /content/DiffusionBlocksPlusPlus

## Install dependencies

Colab already provides CUDA-enabled PyTorch. This installs the project's remaining packages and skips `flash-attn`, which this code does not directly use.

In [ ]:
%pip install 'lightning==2.5.0' 'wandb>=0.22.3' 'transformers==4.52.4' 'datasets==3.5.0' 'huggingface-hub[hf-xet]>=0.31.2' 'hf-transfer>=0.1.9' 'scipy>=1.16.2'
%env USE_TF=0

## Train the model

This creates `logs/` automatically. W&B logging is online in this version, so training metrics appear in your W&B run.

## Sign in to Weights & Biases

Run this cell and complete the W&B authentication prompt before starting training. The training command below runs online (without `--debug`) so metrics are uploaded as training progresses.

In [ ]:
import wandb
wandb.login()

In [ ]:
EPOCHS = 5
BATCH_SIZE = 128
NUM_BLOCKS = 3

!python main.py train cifar100 --model_type dblock \
  --num_blocks {NUM_BLOCKS} --num_epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} --num_workers 2

## Copy a saved checkpoint to Drive

Review the checkpoint paths, then the next cell selects the most recently modified `last.ckpt` from a dblock run and copies it to Drive. If you prefer a best-validation checkpoint, select the corresponding `*.ckpt` path shown above and change `SOURCE` below.

In [ ]:
!find logs -type f -name '*.ckpt' -print

In [ ]:
from pathlib import Path
import shutil

candidates = sorted(
    Path("logs").glob("*dblock*/last.ckpt"),
    key=lambda p: p.stat().st_mtime,
)
if not candidates:
    raise FileNotFoundError("No dblock last.ckpt found under logs/")
SOURCE = candidates[-1]
DESTINATION = Path("/content/drive/MyDrive/dblock-checkpoint.ckpt")
DESTINATION.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(SOURCE, DESTINATION)
print(f"Copied {SOURCE} to {DESTINATION}")

## Compare Euler and midpoint across inference sizes

Use the **same checkpoint**, training block count, seed, and evaluation batch size for every combination. `NUM_BLOCKS` must match training; it is independent of `INFERENCE_SIZES`.

In this implementation, `--num_inference_steps N` creates **N sigma points**, hence **N − 1 integration updates**, followed by a final classification call. Euler uses N denoiser calls; midpoint uses 2(N − 1) + 1. Midpoint evaluates its second slope halfway through each interval; Euler uses only the starting slope. These are equal-grid comparisons, not equal-compute comparisons. More points need not improve classification accuracy.

The primary comparison is Euler versus midpoint. To also evaluate Heun, add `"heun"` to `METHODS`; it uses the same denoiser-call count as midpoint.

The defaults run 10 evaluations: two methods × five sizes. For the denser sweep used in our experiments, use `list(range(2, 11)) + [25, 100]`. Each command creates its own online W&B run in `dblocks-cifar100`; `WANDB_RUN_GROUP` groups the sweep. The CSV below provides an explicit method/size-to-W&B-run-ID mapping.

Results are written to a timestamped Drive directory, so they survive runtime resets and subsequent sweeps do not overwrite them. Model downloads and dataset preparation can make the first evaluation slower; the recorded duration includes process startup and logging, not just inference.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

CHECKPOINT = Path("/content/drive/MyDrive/dblock-checkpoint.ckpt")
NUM_BLOCKS = 3  # Must match the checkpoint's training configuration.
METHODS = ["euler", "midpoint"]  # Optionally add "heun".
INFERENCE_SIZES = [3, 5, 10, 25, 100]
EVAL_BATCH_SIZE = 128
SEED = 42

if not CHECKPOINT.is_file():
    raise FileNotFoundError(f"Upload or copy your checkpoint first: {CHECKPOINT}")
if not METHODS or not INFERENCE_SIZES:
    raise ValueError("Choose at least one method and inference size")
if any(method not in {"euler", "heun", "midpoint"} for method in METHODS):
    raise ValueError("Supported methods: euler, heun, midpoint")
if any(size < 2 for size in INFERENCE_SIZES):
    raise ValueError("Use at least two sigma points for an integration comparison")

SWEEP_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
INFERENCE_RESULTS_DIR = Path("/content/drive/MyDrive/inference_results") / SWEEP_ID
INFERENCE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results: {INFERENCE_RESULTS_DIR}")

### Run the sweep and save metrics

This uses `subprocess` with ordinary Python arguments rather than Colab `!` interpolation. Each evaluation writes a complete `<method>_<size>.log`. The cell extracts the final Lightning `test/acc` and `test/f1` table into `metrics.csv` after **every run**, recording failures as well. Failed runs remain available for inspection and do not prevent the other combinations from running.

The same seed controls initialization across runs; repeat the sweep with other seeds to measure variation. W&B authentication is inherited from the login cell. While a command runs, open its W&B run or log to inspect progress.

In [ ]:
import csv
import os
import re
import subprocess
import sys
import time

METRICS_PATH = INFERENCE_RESULTS_DIR / "metrics.csv"
columns = ["method", "inference_steps", "denoiser_calls", "num_blocks", "seed",
           "checkpoint", "test/acc", "test/f1", "elapsed_seconds", "status",
           "returncode", "wandb_run_id", "log_file"]
records = []
ansi_escape = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")

for method in METHODS:
    for steps in INFERENCE_SIZES:
        log_path = INFERENCE_RESULTS_DIR / f"{method}_{steps}.log"
        command = [
            sys.executable, "-u", "main.py", "test", "cifar100",
            "--model_type", "dblock", "--ckpt_path", str(CHECKPOINT),
            "--num_blocks", str(NUM_BLOCKS), "--num_inference_steps", str(steps),
            "--eval_batch_size", str(EVAL_BATCH_SIZE), "--num_workers", "2",
            "--seed", str(SEED), "--ode", method,
        ]
        env = dict(os.environ, USE_TF="0", WANDB_RUN_GROUP=f"solver-sweep-{SWEEP_ID}",
                   WANDB_TAGS=f"{method},sigma-points-{steps},seed-{SEED}")
        # main.py currently chooses its own UUID for the W&B run; recover it
        # from the log below instead of relying on WANDB_RUN_ID.
        print(f"Running {method}, {steps} sigma points → {log_path.name}", flush=True)
        start = time.perf_counter()
        with log_path.open("w") as log:
            completed = subprocess.run(command, env=env, stdout=log,
                                       stderr=subprocess.STDOUT, check=False)
        elapsed = time.perf_counter() - start
        output = ansi_escape.sub("", log_path.read_text())
        metrics = {}
        for metric in ["test/acc", "test/f1"]:
            values = re.findall(re.escape(metric) + r"\s*[│|]\s*([0-9.eE+\-]+)", output)
            metrics[metric] = float(values[-1]) if values else None
        run_ids = re.findall(r"https://wandb\.ai/[^\s/]+/[^\s/]+/runs/([a-zA-Z0-9]+)", output)
        status = "ok" if completed.returncode == 0 and metrics["test/acc"] is not None else "failed"
        records.append(dict(method=method, inference_steps=steps,
                            denoiser_calls=steps if method == "euler" else 2 * steps - 1,
                            num_blocks=NUM_BLOCKS, seed=SEED, checkpoint=str(CHECKPOINT),
                            **metrics, elapsed_seconds=round(elapsed, 2), status=status,
                            returncode=completed.returncode,
                            wandb_run_id=run_ids[-1] if run_ids else "",
                            log_file=str(log_path)))
        with METRICS_PATH.open("w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=columns)
            writer.writeheader()
            writer.writerows(records)
        print(f"  {status}: test/acc={metrics['test/acc']}, elapsed={elapsed:.1f}s")
        if status != "ok":
            print("\n".join(output.splitlines()[-15:]))

print(f"Saved {len(records)} evaluations to {METRICS_PATH}")

### Inspect and plot the comparison

Accuracy is displayed as a percentage; the CSV stores the original 0–1 values. The horizontal axis counts sigma points, not denoiser calls. Only successful runs are plotted. A single seed is exploratory evidence, not an uncertainty estimate.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

inference_results = pd.read_csv(METRICS_PATH)
display(inference_results)
successful = inference_results.loc[inference_results["status"] == "ok"]
if successful.empty:
    raise RuntimeError(f"No successful evaluations; inspect logs in {INFERENCE_RESULTS_DIR}")
display(successful.pivot(index="inference_steps", columns="method", values="test/acc"))

fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
for method, group in successful.groupby("method"):
    group = group.sort_values("inference_steps")
    ax.plot(group["inference_steps"], group["test/acc"], marker="o", label=method.capitalize())
ticks = sorted(successful["inference_steps"].unique())
ax.set_xscale("log")
ax.set_xticks(ticks, labels=[str(value) for value in ticks])
ax.minorticks_off()
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set(xlabel="Inference sigma points (log scale)", ylabel="Test accuracy",
       title="DiffusionBlocks++: solver comparison")
ax.grid(alpha=0.25)
ax.legend()
for extension in ["png", "pdf"]:
    fig.savefig(INFERENCE_RESULTS_DIR / f"test_accuracy.{extension}", dpi=200)
plt.show()
print(f"Logs, metrics.csv, and chart saved in {INFERENCE_RESULTS_DIR}")

## Running the same experiment on AWS

Use a GPU EC2 instance with a working NVIDIA driver (`nvidia-smi`) and a CUDA-enabled PyTorch environment (`torch.cuda.is_available()` must be `True`). Clone this repository and install the dependencies from the setup section. Activate that environment before starting Jupyter or running commands.

Skip Google Drive mounting and Colab `%cd`/`%pip` commands on EC2: use your local repository directory, `python -m pip install ...` in the shell, and a checkpoint uploaded with `scp` **from your Mac**. In the configuration, change `CHECKPOINT` to the EC2 checkpoint path and `INFERENCE_RESULTS_DIR` to a directory on your persistent EBS volume. Run `wandb login` first. The sweep code uses ordinary Python and works in a Jupyter notebook there too.

For a terminal workflow, use the same `python main.py test ...` arguments inside a `tmux` session; omit the Colab `!` prefix and replace notebook `{VARIABLE}` placeholders with actual values or shell variables. Download results before terminating the instance and stop it when finished to avoid ongoing compute charges.